# Accessing Measurement Archive for Synthetic Dataset
Let's explore the conten of the measurement archive `../data/measurements.jsonl.gz`.
This archive contains raw measurement data for zero-noise extrapolation (ZNE) experiments.
The data is stored in compressed JSON Lines format (`.jsonl.gz`), where each line (3,599,569 in total) represents a single evaluation instance (one circuit, one observable, one repetition).


## 1. Top-Level Structure

Each record has the following structure:

```json
{
  "case_id": "...",
  "args": { ... },
  "moments": [ ... ]
}
```


## 2. Field Descriptions

### 2.1 `case_id`

- Unique identifier for the measurement instance.

---

### 2.2 `args`

Describes the configuration of the experiment.

- **`scale_noise`**: Noise-scaling method used (e.g., `fold_gates_at_random`).
- **`shots`**: Number of measurement shots per execution (typically 10,000).
- **`noise_levels`**: List of noise scale factors (λ values) used in the experiment (typically `[1,2,3,4,5]`).
- **`runtime_provider`**: Backend or simulator used for execution.
- **`observable`**: Measured Pauli observable (e.g., `ZZZZZZZZZ`).
- **`seed`**: Random seed controlling noise scaling and execution variability.
- **`circuit_path`**: Path to the corresponding circuit in `qasm_archive.zip`.
- **`simulate_full_device`**: Whether full-device simulation was used (typically `false`).
- **`ideal_expectation`**: Ideal (noise-free) expectation value of the observable.
- **`num_qubits`**: Number of qubits in the circuit.

---

### 2.3 `moments`

A list of measurement results for different noise scale factors.

Each entry has the form:

```json
{
  "scale_factor": ...,
  "expectation_value": ...,
  "actual_circuit_depth": ...,
  "actual_gate_count": ...
}
```

#### Fields:

- **`scale_factor`**: Noise scaling factor (λ).
- **`expectation_value`**: Measured expectation value at this noise level.
- **`actual_circuit_depth`**: Depth of the executed circuit after scaling.
- **`actual_gate_count`**: Number of gates in the executed circuit.

---

## 3. Interpretation

Each record corresponds to a single noisy experiment instance:
- A specific circuit and observable
- Executed under multiple noise scale factors
- Producing a set of expectation values (`moments`)

These values form the input to extrapolation procedures used in ZNE.

- The sequence of (`scale_factor`, `expectation_value`) pairs defines the noise curve.
- Extrapolation models use this curve to estimate the **zero-noise value**.



Helper functions to read and process the data:

In [1]:
import gzip
import json

def read_jsonl_gz(path, max_records=None):
    """
    Reads a .jsonl.gz file and yields records one by one.

    Parameters
    ----------
    path : str
        Path to the .jsonl.gz file.
    max_records : int | None
        Optional limit on number of records to read.

    Yields
    ------
    dict
        Parsed JSON object per line.
    """
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_records is not None and i >= max_records:
                break
            yield json.loads(line)


def extract_moments(record):
    """
    Extracts scale factors and expectation values from a record.

    Parameters
    ----------
    record : dict
        One JSONL record containing a 'moments' field.

    Returns
    -------
    tuple[list, list]
        (scale_factors, expectation_values)
    """
    moments = record.get("moments", [])

    scale_factors = [m["scale_factor"] for m in moments]
    expectation_values = [m["expectation_value"] for m in moments]

    return scale_factors, expectation_values

Read first record:

In [2]:
archive_file_name = "../data/measurements.jsonl.gz"
first_observation = next(read_jsonl_gz(archive_file_name))

print(json.dumps(first_observation, indent=2))

{
  "case_id": "case_0000199",
  "args": {
    "scale_noise": "fold_gates_at_random",
    "shots": 10000,
    "noise_levels": [
      1,
      2,
      3,
      4,
      5
    ],
    "runtime_provider": "ibm_algiers",
    "observable": "ZZZZZZZZZZ",
    "seed": 200,
    "circuit_path": "10qubits_0.05bin_width_100circuits_per_bin_10circuit_depth/circuit_-0.05_14.qasm3.qasm",
    "simulate_full_device": false,
    "ideal_expectation": -0.00021916657718728649,
    "num_qubits": 10
  },
  "moments": [
    {
      "expectation_value": -0.0122,
      "actual_gate_count": 967,
      "scale_factor": 1,
      "actual_circuit_depth": 247
    },
    {
      "expectation_value": 0.0158,
      "actual_gate_count": 2047,
      "scale_factor": 2,
      "actual_circuit_depth": 538
    },
    {
      "expectation_value": -0.0002,
      "actual_gate_count": 3235,
      "scale_factor": 3,
      "actual_circuit_depth": 825
    },
    {
      "expectation_value": 0.0156,
      "actual_gate_count": 4315,
  

Extract scale factors and expectation values:

In [3]:
scale_factors, expectation_values = extract_moments(first_observation)
print(f"scale_factors: {scale_factors}")
print(f"expectation_values: {expectation_values}")


scale_factors: [1, 2, 3, 4, 5]
expectation_values: [-0.0122, 0.0158, -0.0002, 0.0156, 0.0032]


Iterate over multiple records:

In [4]:
for record in read_jsonl_gz(archive_file_name, max_records=3):
    case_id = record.get("case_id")
    observable = record.get("args").get("observable")

    scale_factors, expectation_values = extract_moments(record)

    print(f"case_id: {case_id}")
    print(f"observable: {observable}")
    print(f"scale_factors: {scale_factors}")
    print(f"expectation_values: {expectation_values}")
    print("-" * 70)

case_id: case_0000199
observable: ZZZZZZZZZZ
scale_factors: [1, 2, 3, 4, 5]
expectation_values: [-0.0122, 0.0158, -0.0002, 0.0156, 0.0032]
----------------------------------------------------------------------
case_id: case_0002848
observable: ZZZZZZZZZZ
scale_factors: [1, 2, 3, 4, 5]
expectation_values: [-0.0236, 0.01, 0.0064, -0.012, -0.0076]
----------------------------------------------------------------------
case_id: case_0003798
observable: ZZZZZZZZZZ
scale_factors: [1, 2, 3, 4, 5]
expectation_values: [-0.0632, -0.0312, -0.027, -0.0268, -0.012]
----------------------------------------------------------------------


# Accessing Measurement Archive for Real Quantum Device

The measurements stored in `measurements_real.jsonl.gz` (collected from a real quantum device) can be accessed in the same way:

In [5]:
archive_file_name = "../data/measurements_real.jsonl.gz"
first_observation = next(read_jsonl_gz(archive_file_name))

print(json.dumps(first_observation, indent=2))

{
  "case_id": "ghz_alg_30__X_10000_shots_seed_42_group_1",
  "args": {
    "shots": 10000,
    "ideal_expectation": 1.0,
    "num_qubits": 30,
    "runtime_provider": "ibm_kingston",
    "circuit_path": "ghz_alg_30.qasm",
    "observable": "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
  },
  "moments": [
    {
      "actual_circuit_depth": 94,
      "scale_factor": 1.0,
      "expectation_value": 0.0268,
      "actual_gate_count": 297
    },
    {
      "actual_circuit_depth": 136,
      "scale_factor": 1.3,
      "expectation_value": -0.0204,
      "actual_gate_count": 367
    },
    {
      "actual_circuit_depth": 182,
      "scale_factor": 1.6,
      "expectation_value": -0.0436,
      "actual_gate_count": 465
    }
  ]
}


In [6]:
for record in read_jsonl_gz(archive_file_name, max_records=3):
    case_id = record.get("case_id")
    observable = record.get("args").get("observable")

    scale_factors, expectation_values = extract_moments(record)

    print(f"case_id: {case_id}")
    print(f"observable: {observable}")
    print(f"scale_factors: {scale_factors}")
    print(f"expectation_values: {expectation_values}")
    print("-" * 70)

case_id: ghz_alg_30__X_10000_shots_seed_42_group_1
observable: XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
scale_factors: [1.0, 1.3, 1.6]
expectation_values: [0.0268, -0.0204, -0.0436]
----------------------------------------------------------------------
case_id: ghz_alg_30__X_10000_shots_seed_42_group_2
observable: XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
scale_factors: [1.0, 1.3, 1.6]
expectation_values: [0.0306, -0.0026, -0.0412]
----------------------------------------------------------------------
case_id: ghz_alg_30__X_10000_shots_seed_42_group_3
observable: XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
scale_factors: [1.0, 1.3, 1.6]
expectation_values: [0.0478, 0.0048, -0.0608]
----------------------------------------------------------------------
